In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
import pickle
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report

SAVE_DIR = "../datasets/preprocessed"
MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

NUM_FEATURES = 68   # jumlah fitur setelah drop zero variance
NUM_CLASSES = 11    # jumlah kelas serangan + benign

# gunakan GPU jika tersedia untuk mempercepat training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 10
print("Device:", DEVICE)

Device: cuda


In [2]:
class ANN(nn.Module):
    """
    Arsitektur ANN untuk klasifikasi network intrusion detection.

    Struktur: Input(68) -> 256 -> 128 -> 64 -> Output(11)

    Justifikasi arsitektur:
    - 3 hidden layer: cukup untuk menangkap pola kompleks di network traffic
      tanpa overfitting berlebihan pada data tabular
    - Ukuran layer menurun (256->128->64): gradually compressing representation
      dari fitur umum ke fitur spesifik per kelas
    - ReLU: menghindari vanishing gradient dibanding sigmoid/tanh,
      komputasi ringan, standard untuk hidden layer
    - Dropout 0.3: regularisasi untuk mencegah overfitting,
      30% neuron dinonaktifkan secara random tiap forward pass
    - Output tanpa aktivasi (raw logits): karena CrossEntropyLoss
      sudah include Softmax secara internal
    """
    def __init__(self, input_dim, num_classes):
        super(ANN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

model = ANN(NUM_FEATURES, NUM_CLASSES)
print(model)
print(f"\nTotal parameter: {sum(p.numel() for p in model.parameters()):,}")

ANN(
  (network): Sequential(
    (0): Linear(in_features=68, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=64, out_features=11, bias=True)
  )
)

Total parameter: 59,531


In [3]:
def make_dataloader(X, y, batch_size=1024, shuffle=True):
    """
    Buat DataLoader dari numpy array.
    Batch size 1024 dipilih sebagai trade-off antara
    kecepatan training dan stabilitas gradient.
    """
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            output = model(X_batch)
            preds = torch.argmax(output, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())
    return np.array(all_preds), np.array(all_labels)

In [4]:
# load label encoder untuk nama kelas
with open(f"{SAVE_DIR}/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# load data node 1: Brute Force (FTP-Patator, SSH-Patator)
X_train1 = np.load(f"{SAVE_DIR}/node1_X_train.npy")
X_test1 = np.load(f"{SAVE_DIR}/node1_X_test.npy")
y_train1 = np.load(f"{SAVE_DIR}/node1_y_train.npy")
y_test1 = np.load(f"{SAVE_DIR}/node1_y_test.npy")

train_loader1 = make_dataloader(X_train1, y_train1)
test_loader1 = make_dataloader(X_test1, y_test1, shuffle=False)

# inisialisasi model, optimizer, dan loss function
# Adam dipilih karena adaptive learning rate, lebih robust untuk
# dataset imbalanced dibanding SGD biasa
# lr=1e-3 adalah nilai default Adam yang terbukti stabil
model1 = ANN(NUM_FEATURES, NUM_CLASSES).to(DEVICE)
optimizer1 = torch.optim.Adam(model1.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("=== Training Node 1: Brute Force ===")
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model1, train_loader1, optimizer1, criterion, DEVICE)
    print(f"Epoch {epoch}/{EPOCHS} - Loss: {loss:.4f}")

preds1, labels1 = evaluate(model1, test_loader1, DEVICE)
present_classes1 = np.unique(labels1)
present_names1 = le.classes_[present_classes1]

print("\n=== Evaluasi Node 1 ===")
print(classification_report(
    labels1, preds1,
    labels=present_classes1,
    target_names=present_names1,
    digits=4
))

torch.save(model1.state_dict(), f"{MODEL_DIR}/node1_standalone.pth")
print("Model node 1 tersimpan.")

=== Training Node 1: Brute Force ===
Epoch 1/10 - Loss: 0.1420
Epoch 2/10 - Loss: 0.0122
Epoch 3/10 - Loss: 0.0087
Epoch 4/10 - Loss: 0.0068
Epoch 5/10 - Loss: 0.0061
Epoch 6/10 - Loss: 0.0060
Epoch 7/10 - Loss: 0.0054
Epoch 8/10 - Loss: 0.0055
Epoch 9/10 - Loss: 0.0046
Epoch 10/10 - Loss: 0.0050

=== Evaluasi Node 1 ===
              precision    recall  f1-score   support

      BENIGN     0.9998    0.9998    0.9998     86363
 FTP-Patator     0.9919    0.9975    0.9947      1587
 SSH-Patator     0.9914    0.9830    0.9872      1179

    accuracy                         0.9995     89129
   macro avg     0.9944    0.9934    0.9939     89129
weighted avg     0.9995    0.9995    0.9995     89129

Model node 1 tersimpan.


In [5]:
# load data node 2: DoS attacks
X_train2 = np.load(f"{SAVE_DIR}/node2_X_train.npy")
X_test2 = np.load(f"{SAVE_DIR}/node2_X_test.npy")
y_train2 = np.load(f"{SAVE_DIR}/node2_y_train.npy")
y_test2 = np.load(f"{SAVE_DIR}/node2_y_test.npy")

train_loader2 = make_dataloader(X_train2, y_train2)
test_loader2 = make_dataloader(X_test2, y_test2, shuffle=False)

model2 = ANN(NUM_FEATURES, NUM_CLASSES).to(DEVICE)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

print("=== Training Node 2: DoS ===")
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model2, train_loader2, optimizer2, criterion, DEVICE)
    print(f"Epoch {epoch}/{EPOCHS} - Loss: {loss:.4f}")

preds2, labels2 = evaluate(model2, test_loader2, DEVICE)
present_classes2 = np.unique(labels2)
present_names2 = le.classes_[present_classes2]

print("\n=== Evaluasi Node 2 ===")
print(classification_report(
    labels2, preds2,
    labels=present_classes2,
    target_names=present_names2,
    digits=4
))

torch.save(model2.state_dict(), f"{MODEL_DIR}/node2_standalone.pth")
print("Model node 2 tersimpan.")

=== Training Node 2: DoS ===
Epoch 1/10 - Loss: 0.1430
Epoch 2/10 - Loss: 0.0376
Epoch 3/10 - Loss: 0.0314
Epoch 4/10 - Loss: 0.0289
Epoch 5/10 - Loss: 0.0272
Epoch 6/10 - Loss: 0.0263
Epoch 7/10 - Loss: 0.0252
Epoch 8/10 - Loss: 0.0244
Epoch 9/10 - Loss: 0.0245
Epoch 10/10 - Loss: 0.0230

=== Evaluasi Node 2 ===
                  precision    recall  f1-score   support

          BENIGN     0.9995    0.9877    0.9935     87936
   DoS GoldenEye     0.9956    0.9869    0.9912      2059
        DoS Hulk     0.9778    0.9994    0.9885     46025
DoS Slowhttptest     0.9520    0.9909    0.9710      1100
   DoS slowloris     0.9844    0.9793    0.9818      1159

        accuracy                         0.9915    138279
       macro avg     0.9819    0.9888    0.9852    138279
    weighted avg     0.9917    0.9915    0.9916    138279

Model node 2 tersimpan.


In [6]:
# load data node 3: DDoS, Web Attack, Bot, PortScan
X_train3 = np.load(f"{SAVE_DIR}/node3_X_train.npy")
X_test3 = np.load(f"{SAVE_DIR}/node3_X_test.npy")
y_train3 = np.load(f"{SAVE_DIR}/node3_y_train.npy")
y_test3 = np.load(f"{SAVE_DIR}/node3_y_test.npy")

train_loader3 = make_dataloader(X_train3, y_train3)
test_loader3 = make_dataloader(X_test3, y_test3, shuffle=False)

# node 3 pakai class weighting karena Bot (0.22%) dan Web Attack (0.25%)
# sangat minoritas dibanding BENIGN, DDoS, PortScan
# tanpa weighting, model mengabaikan kelas minoritas ini
# Bot=15x dan Web Attack=12x dipilih setelah tuning manual:
# - balanced (89x/80x) terlalu agresif, precision anjlok
# - 25x terlalu tinggi untuk Bot, precision 0.17
# - 15x/12x memberikan trade-off terbaik macro F1=0.8594
custom_weights = torch.ones(NUM_CLASSES)
custom_weights[1] = 15.0   # Bot: index 1
custom_weights[10] = 12.0  # Web Attack: index 10
criterion_weighted = nn.CrossEntropyLoss(weight=custom_weights.to(DEVICE))

model3 = ANN(NUM_FEATURES, NUM_CLASSES).to(DEVICE)
optimizer3 = torch.optim.Adam(model3.parameters(), lr=1e-3)

print("=== Training Node 3: DDoS/Web/Bot/PortScan ===")
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model3, train_loader3, optimizer3, criterion_weighted, DEVICE)
    print(f"Epoch {epoch}/{EPOCHS} - Loss: {loss:.4f}")

preds3, labels3 = evaluate(model3, test_loader3, DEVICE)
present_classes3 = np.unique(labels3)
present_names3 = le.classes_[present_classes3]

print("\n=== Evaluasi Node 3 ===")
print(classification_report(
    labels3, preds3,
    labels=present_classes3,
    target_names=present_names3,
    digits=4
))

torch.save(model3.state_dict(), f"{MODEL_DIR}/node3_standalone.pth")
print("Model node 3 tersimpan.")

=== Training Node 3: DDoS/Web/Bot/PortScan ===
Epoch 1/10 - Loss: 0.1917
Epoch 2/10 - Loss: 0.0592
Epoch 3/10 - Loss: 0.0501
Epoch 4/10 - Loss: 0.0467
Epoch 5/10 - Loss: 0.0443
Epoch 6/10 - Loss: 0.0422
Epoch 7/10 - Loss: 0.0420
Epoch 8/10 - Loss: 0.0399
Epoch 9/10 - Loss: 0.0392
Epoch 10/10 - Loss: 0.0385

=== Evaluasi Node 3 ===
              precision    recall  f1-score   support

      BENIGN     0.9990    0.9813    0.9901    116397
         Bot     0.1626    0.8798    0.2745       391
        DDoS     0.9987    0.9985    0.9986     25605
    PortScan     0.9997    0.9993    0.9995     31761
  Web Attack     0.5393    0.9748    0.6944       436

    accuracy                         0.9869    174590
   macro avg     0.7399    0.9667    0.7914    174590
weighted avg     0.9960    0.9869    0.9907    174590

Model node 3 tersimpan.


In [ ]:
# load data node 3: DDoS, Web Attack, Bot, PortScan
X_train3 = np.load(f"{SAVE_DIR}/node3_X_train.npy")
X_test3 = np.load(f"{SAVE_DIR}/node3_X_test.npy")
y_train3 = np.load(f"{SAVE_DIR}/node3_y_train.npy")
y_test3 = np.load(f"{SAVE_DIR}/node3_y_test.npy")

train_loader3 = make_dataloader(X_train3, y_train3)
test_loader3 = make_dataloader(X_test3, y_test3, shuffle=False)

# node 3 pakai class weighting karena Bot (0.22%) dan Web Attack (0.25%)
# sangat minoritas dibanding kelas dominan (BENIGN 66%, PortScan 18%)
# class weighting memberi penalti lebih tinggi pada misklasifikasi
# kelas minoritas sehingga gradient tidak didominasi kelas mayoritas
#
# nilai weight ditentukan melalui grid search manual tiga konfigurasi:
# sklearn balanced (89x/80x): macro F1=0.7389, precision terlalu rendah
# custom 15x/12x:             macro F1=0.8594, trade-off terbaik
# custom 25x/12x:             macro F1=0.7939, Bot precision anjlok ke 0.17
#
# kesimpulan: Bot=15x, Web Attack=12x dipilih karena macro F1 tertinggi
custom_weights = torch.ones(NUM_CLASSES)
custom_weights[1] = 15.0   # Bot: index 1
custom_weights[10] = 12.0  # Web Attack: index 10
criterion_weighted = nn.CrossEntropyLoss(weight=custom_weights.to(DEVICE))
custom_weights = torch.ones(NUM_CLASSES)
custom_weights[1] = 15.0   # Bot: index 1
custom_weights[10] = 12.0  # Web Attack: index 10
criterion_weighted = nn.CrossEntropyLoss(weight=custom_weights.to(DEVICE))

model3 = ANN(NUM_FEATURES, NUM_CLASSES).to(DEVICE)
optimizer3 = torch.optim.Adam(model3.parameters(), lr=1e-3)

print("=== Training Node 3: DDoS/Web/Bot/PortScan ===")
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model3, train_loader3, optimizer3, criterion_weighted, DEVICE)
    print(f"Epoch {epoch}/{EPOCHS} - Loss: {loss:.4f}")

preds3, labels3 = evaluate(model3, test_loader3, DEVICE)
present_classes3 = np.unique(labels3)
present_names3 = le.classes_[present_classes3]

print("\n=== Evaluasi Node 3 ===")
print(classification_report(
    labels3, preds3,
    labels=present_classes3,
    target_names=present_names3,
    digits=4
))

torch.save(model3.state_dict(), f"{MODEL_DIR}/node3_standalone.pth")
print("Model node 3 tersimpan.")

=== Training Node 3: DDoS/Web/Bot/PortScan ===
Epoch 1/10 - Loss: 0.1920
Epoch 2/10 - Loss: 0.0583
Epoch 3/10 - Loss: 0.0498
Epoch 4/10 - Loss: 0.0458
Epoch 5/10 - Loss: 0.0423
Epoch 6/10 - Loss: 0.0423
Epoch 7/10 - Loss: 0.0409
Epoch 8/10 - Loss: 0.0393
Epoch 9/10 - Loss: 0.0393
Epoch 10/10 - Loss: 0.0381

=== Evaluasi Node 3 ===
              precision    recall  f1-score   support

      BENIGN     0.9985    0.9918    0.9952    116397
         Bot     0.3484    0.7315    0.4719       391
        DDoS     0.9991    0.9984    0.9988     25605
    PortScan     0.9997    0.9993    0.9995     31761
  Web Attack     0.5220    0.9794    0.6810       436

    accuracy                         0.9936    174590
   macro avg     0.7736    0.9401    0.8293    174590
weighted avg     0.9962    0.9936    0.9945    174590

Model node 3 tersimpan.
